# 968. Binary Tree Cameras

## Topic Alignment
- This problem models sensor placement optimization, which appears in IoT network design, surveillance system optimization, and coverage problems in distributed systems.
- The multi-state tree DP pattern is relevant for hierarchical resource allocation, optimal placement problems in ML infrastructure (e.g., cache placement, model deployment), and monitoring system design.
- Understanding greedy + DP hybrid approaches is crucial for optimization problems in production ML systems.

## Metadata 摘要
- Source: https://leetcode.com/problems/binary-tree-cameras/
- Tags: Tree, Dynamic Programming, DFS, Greedy, Binary Tree
- Difficulty: Hard
- Priority: High

## Problem Statement 原题描述
You are given the `root` of a binary tree. We install cameras on the tree nodes where each camera at a node can monitor its parent, itself, and its immediate children.

Return the minimum number of cameras needed to monitor all nodes of the tree.

**Constraints**:
- The number of nodes in the tree is in the range `[1, 1000]`.
- `Node.val == 0`

## Progressive Hints
- Hint 1: A camera covers itself, its parent, and its children (range = 1).
- Hint 2: Greedy insight: prefer placing cameras on parent nodes rather than leaf nodes (one camera can cover more nodes).
- Hint 3: Use post-order DFS to process children before parent.
- Hint 4: Define three states for each node: (0) not covered, (1) covered but no camera, (2) has camera.
- Hint 5: If a child is not covered (state 0), parent must place a camera.
- Hint 6: If both children are covered (state 1), parent can choose not to place camera (let grandparent cover it).
- Hint 7: Special case: if root returns state 0 (not covered), add one more camera at root.
- Hint 8: The recurrence combines greedy (when forced to place camera) with DP (optimal choice when free).

## Solution Overview
Use **Tree DP with 3 states** combined with **greedy strategy**.

**State Definition**: Each node can be in one of three states:
- **State 0**: Node is **not covered** (needs coverage from ancestor)
- **State 1**: Node is **covered but has no camera** (covered by child's camera)
- **State 2**: Node **has a camera** (covers itself, parent, children)

**Greedy Insight**: Place cameras as high as possible (closer to root) to maximize coverage.

**Recurrence Logic**:
```python
if any child is state 0 (not covered):
    parent must have camera → state 2
elif any child has camera (state 2):
    parent is covered → state 1
else:  # both children are state 1 (covered, no camera)
    parent chooses not to place camera → state 0
    (let grandparent place camera)
```

**Base case**: Null nodes are considered state 1 (covered, no camera needed)

**Root handling**: If root is state 0 after DFS, add one camera at root.

## Detailed Explanation

### Problem Understanding

**Camera coverage**:
- A camera at node X covers: parent(X), X, left_child(X), right_child(X)
- Coverage radius = 1 (one edge away)

**Goal**: Minimize number of cameras while ensuring every node is covered.

---

### Why Greedy Doesn't Work Alone

Pure greedy (e.g., always place at leaves) is suboptimal:
```
    A
   / \\
  B   C
 / \\
D   E
```

- Greedy at leaves: cameras at D, E → 2 cameras (C uncovered, need 3 total)
- Optimal: camera at B → 1 camera covers B, A, D, E (still need 1 for C)

**Key insight**: Placing camera at parent of leaves is better than at leaves.

---

### State Design

**Three states capture all possibilities**:

1. **State 0 (NOT_COVERED)**: 
   - Node is not covered by any camera
   - Needs ancestor to place camera
   - Triggers parent to place camera

2. **State 1 (COVERED_NO_CAMERA)**:
   - Node is covered (by child's or parent's camera)
   - Does not have camera itself
   - Safe state

3. **State 2 (HAS_CAMERA)**:
   - Node has a camera
   - Covers itself, parent, and children
   - Increments camera count

---

### Recurrence Logic (Post-Order DFS)

**Process children first, then decide for parent**:

```python
left_state = dfs(node.left)
right_state = dfs(node.right)

# Case 1: Any child not covered → MUST place camera here
if left_state == 0 or right_state == 0:
    cameras += 1
    return 2  # HAS_CAMERA

# Case 2: Any child has camera → this node is covered
if left_state == 2 or right_state == 2:
    return 1  # COVERED_NO_CAMERA

# Case 3: Both children covered but no camera
# Don't place camera here, let parent handle it
return 0  # NOT_COVERED
```

**Why Case 3 returns 0?**
- Both children are state 1 (covered, no camera)
- Current node is not covered by children
- **Greedy choice**: Don't place camera here, let parent place it
- Parent's camera will cover: parent, current node, sibling
- This is optimal because one camera covers more nodes

---

### Why Null Nodes Return State 1?

Null nodes should not trigger camera placement:
- If null returns 0 (not covered), parent would always place camera (wrong)
- If null returns 1 (covered), parent treats it as safe (correct)
- Null returning 1 means "this subtree is handled, don't worry"

---

### Root Special Case

After DFS, if root returns state 0 (not covered):
- No parent exists to place camera
- Must place camera at root
- Add 1 to total cameras

---

### Example Walkthrough

Tree:
```
      0
     / \\
    0   0
         \\
          0
           \\
            0
```

**Bottom-up computation**:

Node (leaf, rightmost):
- left = null (state 1), right = null (state 1)
- Both children state 1 → return 0 (NOT_COVERED)

Parent of leaf:
- left = null (state 1), right = 0 (NOT_COVERED)
- Right child not covered → **place camera**, return 2 (HAS_CAMERA)
- cameras = 1

Parent of above:
- left = null (state 1), right = 2 (HAS_CAMERA)
- Right child has camera → this node covered, return 1 (COVERED_NO_CAMERA)

Left child of root:
- left = null (state 1), right = null (state 1)
- Both state 1 → return 0 (NOT_COVERED)

Root:
- left = 0 (NOT_COVERED), right = 1 (COVERED_NO_CAMERA)
- Left child not covered → **place camera**, return 2 (HAS_CAMERA)
- cameras = 2

**Answer**: 2 cameras (one at root, one at node covering rightmost leaf)

---

### Why This Is Optimal

**Greedy principle**: Place cameras as high as possible
- Placing at parent covers more nodes than at child
- Post-order DFS ensures we see children before parent
- State 0 from child forces parent to place camera (necessary)
- State 1 from both children allows parent to defer (greedy optimization)

**DP aspect**: Optimal substructure
- Optimal solution for subtree determines parent's decision
- Three states capture all reachable configurations
- Recurrence ensures global optimality

---

### Mathematical Intuition

For a tree with height h:
- Naive: place camera at every node → n cameras
- Greedy: place at every other level → ~n/2 cameras
- Optimal: this algorithm → ~n/3 cameras (each camera covers 3-4 nodes)

The algorithm achieves near-optimal coverage by maximizing overlap.

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| Tree DP (3 states) | O(n) | O(h) | Optimal, one pass |
| Greedy from leaves | O(n) | O(h) | Suboptimal, may use more cameras |
| BFS + marking | O(n) | O(n) | More complex, harder to optimize |
| Backtracking | O(2^n) | O(h) | Try all combinations, impractical |

In [ ]:
# Definition for a binary tree node.
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

class Solution:
    def minCameraCover(self, root: TreeNode) -> int:
        """
        Minimum cameras to cover all nodes using Tree DP.
        
        States:
        0 - NOT_COVERED: node is not covered
        1 - COVERED_NO_CAMERA: node is covered but has no camera
        2 - HAS_CAMERA: node has a camera
        
        Time: O(n) - visit each node once
        Space: O(h) - recursion stack
        """
        self.cameras = 0
        
        # State constants for clarity
        NOT_COVERED = 0
        COVERED_NO_CAMERA = 1
        HAS_CAMERA = 2
        
        def dfs(node):
            """
            Returns state of current node after processing subtree.
            Side effect: increments self.cameras when placing camera.
            """
            if not node:
                # Null nodes are considered covered
                # (don't trigger parent to place camera)
                return COVERED_NO_CAMERA
            
            # Post-order: process children first
            left_state = dfs(node.left)
            right_state = dfs(node.right)
            
            # Case 1: Any child not covered → MUST place camera here
            if left_state == NOT_COVERED or right_state == NOT_COVERED:
                self.cameras += 1
                return HAS_CAMERA
            
            # Case 2: Any child has camera → current node is covered
            if left_state == HAS_CAMERA or right_state == HAS_CAMERA:
                return COVERED_NO_CAMERA
            
            # Case 3: Both children covered but no camera
            # Greedy: don't place camera, let parent cover this node
            return NOT_COVERED
        
        root_state = dfs(root)
        
        # Special case: if root is not covered, place camera at root
        if root_state == NOT_COVERED:
            self.cameras += 1
        
        return self.cameras

In [ ]:
# Test cases
def build_tree(values):
    """Build binary tree from level-order list (None for null nodes)"""
    if not values:
        return None
    nodes = [TreeNode(v) if v is not None else None for v in values]
    kids = nodes[::-1]
    root = kids.pop()
    for node in nodes:
        if node:
            if kids: node.left = kids.pop()
            if kids: node.right = kids.pop()
    return root

tests = [
    ([0,0,None,0,0], 1),              # Camera at root
    ([0,0,None,0,None,0,None,None,0], 2),  # Linear tree
    ([0], 1),                          # Single node
    ([0,0,0], 1),                      # Three nodes, camera at root
    ([0,0,0,None,None,None,0], 2),    # Skewed tree
]

solver = Solution()
for values, expected in tests:
    root = build_tree(values)
    result = solver.minCameraCover(root)
    assert result == expected, f"Failed for {values}: got {result}, expected {expected}"
print('All tests passed.')

## Complexity Analysis
- **Time**: O(n) where n = number of nodes
  - Each node visited exactly once in post-order DFS
  - Constant work per node (state comparison and update)
- **Space**: O(h) where h = tree height
  - Recursion call stack depth equals height
  - O(log n) for balanced tree
  - O(n) worst case for skewed tree
  - No additional data structures needed

## Edge Cases & Pitfalls
- **Single node**: Need 1 camera (root not covered after DFS)
- **Two nodes**: 1 camera at root covers both
- **Linear tree**: Need approximately n/3 cameras
- **Complete binary tree**: Most efficient coverage, ~n/3 cameras
- **Skewed tree**: Similar to linear array problem
- **Common mistake**: Making null nodes return state 0 (causes over-placement)
- **Common mistake**: Forgetting root special case (root not covered)
- **Common mistake**: Using greedy only (placing at leaves is suboptimal)
- **State confusion**: Understanding when to return each state is crucial
- **Off-by-one**: Forgetting to add 1 when root is state 0

## Follow-up Variants
- **Coverage radius k**: Cameras cover k edges away (generalize to k > 1)
- **Weighted nodes**: Different costs for placing cameras at different nodes
- **Multiple types**: Different camera types with different costs and coverage
- **General tree**: Extend to n-ary trees
- **Graph version**: Minimum vertex cover or dominating set on graphs
- **Dynamic updates**: Add/remove nodes, update minimum cameras efficiently
- **Partial coverage**: Cover at least k% of nodes with minimum cameras
- **Probabilistic**: Cameras have probability of failure, ensure redundancy

## Takeaways
- **Multi-state tree DP** is powerful for complex optimization on trees.
- **Greedy + DP hybrid**: greedy insights guide state transitions, DP ensures optimality.
- **State 0 (not covered) propagates urgency**: forces ancestor to act.
- **State 1 (covered, no camera) is safety**: allows flexibility for ancestors.
- **State 2 (has camera) provides coverage**: influences parent's decision.
- **Null node handling** is critical: wrong choice breaks the algorithm.
- **Root special case** is common in tree DP: no parent to rely on.
- Post-order DFS is natural when child states determine parent action.
- This problem pattern extends to sensor placement, monitoring, and coverage problems.
- Understanding **why greedy works** (when combined with DP) is key to solving hard problems.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 337 | House Robber III | Tree DP with 2 states |
| LC 979 | Distribute Coins in Binary Tree | Tree DP, greedy |
| LC 1026 | Maximum Difference Between Node and Ancestor | Tree DP with max/min |
| LC 2421 | Number of Good Paths | Union-Find on tree |
| LC 1245 | Tree Diameter | Tree DP pattern |
| - | Vertex Cover (Graph Theory) | NP-hard on general graphs |